# Poetry · Comparing authors

Short public-domain poems with author labels. Compare author-specific concepts and inspect their activating passages.

## Install ConceptExplorer from GitHub

Set `GITHUB_REPO` to your published repository (and `PACKAGE_SUBDIR` to the folder containing `pyproject.toml` if the package is in a monorepo). Use a release tag instead of `main` for stable results.


In [ ]:
import sys, subprocess

GITHUB_REPO = "YOUR_ORG/YOUR_REPO"  # Change to the actual GitHub repository
GIT_REF = "main"                  # Or a tagged release
PACKAGE_SUBDIR = ""               # Example: "packages/uce-concept-explorer"

if GITHUB_REPO == "YOUR_ORG/YOUR_REPO":
    raise ValueError("Set GITHUB_REPO to the repository hosting uce-concept-explorer")

url = f"uce-concept-explorer @ git+https://github.com/{GITHUB_REPO}.git@{GIT_REF}"
if PACKAGE_SUBDIR:
    url += f"#subdirectory={PACKAGE_SUBDIR}"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", url])
from uce_concept_explorer import ConceptExplorer, load_metadata


## 1. Install and load UCE-Gemma

Select **Runtime → Change runtime type → T4 GPU** in Colab. This notebook downloads the model weights only for encoding. Subsequent analysis is CPU-only. The Hugging Face base model may require an access token in Colab Secrets (`HF_TOKEN`).

In [ ]:
%pip -q install -U huggingface_hub datasets scipy pandas scikit-learn matplotlib wordcloud
from huggingface_hub import snapshot_download
from pathlib import Path
MODEL_ID = "klpeng/uce-gemma-80k"
model_dir = snapshot_download(MODEL_ID)
import subprocess, sys
req = Path(model_dir) / "requirements.txt"
if req.exists():
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)])

In [ ]:
import os, sys, torch, numpy as np
from scipy import sparse
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
    if token: os.environ['HF_TOKEN'] = token
except Exception: pass
sys.path.insert(0, model_dir)
from uce_gemma import UCEGemma
model = UCEGemma.from_pretrained(model_dir, device='cuda' if torch.cuda.is_available() else 'cpu')
print('Model loaded.')

## Load short poems

In [ ]:
from datasets import load_dataset
import pandas as pd

ds = load_dataset('yoonholee/poetry-greats-public-domain',split='train')
print(ds.column_names)
ds = ds.filter(lambda r: 20 <= r['word_count'] <= 150)
df = ds.to_pandas()
# Choose authors with enough short poems for relatively stable comparisons.
authors_keep = df.author.value_counts().loc[lambda s: s>=30].head(5).index
if len(authors_keep) == 0: raise ValueError('No authors meet the minimum poem count')
sample = pd.concat([df[df.author==a].sample(n=min(100,(df.author==a).sum()),random_state=42) for a in authors_keep]).reset_index(drop=True)
texts = sample.poem_text.astype(str).tolist()
labels = sample.author.to_numpy()
print(sample.author.value_counts())

## 2. Encode once

Batch size is deliberately small for a 16 GB T4. Cache the sparse scores to avoid rerunning the GPU. Documents are encoded with the same ordering as `texts`.

In [ ]:
THRESHOLD = 0.92
BATCH_SIZE = 2
chunks = []
for i in range(0, len(texts), BATCH_SIZE):
    with torch.inference_mode():
        chunk = model.encode(list(texts[i:i+BATCH_SIZE]))
    chunks.append(sparse.csr_matrix(chunk))
    if i % 50 == 0: print(f'{i}/{len(texts)}', end='\r')
X = sparse.vstack(chunks, format='csr')
sparse.save_npz('uce_scores.npz', X)
print('Scores:', X.shape, 'nnz:', X.nnz)
# Later sessions: X = sparse.load_npz('uce_scores.npz')

## Explore concepts (CPU only)

The implementation is imported from GitHub; only dataset-specific analysis appears here. The binary threshold applies to all analyses.


In [ ]:
# Metadata-only loader: no weights are downloaded by this call.
metadata = load_metadata(MODEL_ID)
E = ConceptExplorer(X, texts, metadata, threshold=THRESHOLD)
print('Encoded texts:', len(texts), 'active features:', int((E.counts>0).sum()))
display(E.prevalent(top_n=15))

**Optional fidelity filtering.** The UCE model card describes F1 based on up to 30 positive examples and 30 sparse zero examples per concept; it is an *automated judgment* metric, not corpus-calibrated precision or recall.

In [ ]:
print('Fidelity columns:', [c for c in E.table.columns if c in ('f1','precision','recall','accuracy')])
# Uncomment if available:
# E.set_fidelity_filter(min_fidelity=0.8, field='f1')
# display(E.prevalent(top_n=10))

## Compare poets

In [ ]:
for author, table in E.by_category(labels, top_n=12, multiplicative=False).items():
    print(author)
    display(table)
concept_id = int(E.prevalent(top_n=1).iloc[0].concept_id)
display(E.examples(concept_id, n=5))

## Optional: word cloud of enriched concept labels

Word sizes reflect **positive conservative lower bounds** on author-vs-rest prevalence differences; full multiword UCE labels remain intact.

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

author = list(E.by_category(labels))[0]
table = E.by_category(labels, top_n=45, multiplicative=False, min_count=3)[author]
weights = {str(row.concept_label): max(0.0, float(row.rank_bound))
           for row in table.itertuples() if row.rank_bound > 0}
if weights:
    wc = WordCloud(width=1000, height=600, background_color='white',
                   collocations=False, prefer_horizontal=1).generate_from_frequencies(weights)
    plt.figure(figsize=(12, 6)); plt.imshow(wc); plt.axis('off')
    plt.title(f'Enriched concepts: {author}'); plt.show()
else:
    print('No positively enriched concepts pass this confidence-bound ranking.')

## Optional: continuous outcome

Here the target is poem word count (illustrative). The concept-level univariate \(R^2\) is descriptive, not causal.

In [ ]:
display(E.predictive(sample.word_count.to_numpy(dtype=float), kind='continuous', top_n=15))

## Optional continuous prediction

Predict poem length from concepts (illustrative, not a substantive scientific outcome). Use any aligned continuous target instead.


In [ ]:
# Optional, if your DataFrame contains word counts for the sampled poems:
# reg, metrics, selected = E.regressor(np.asarray(sample["word_count"], float), n_concepts=5)
# print(metrics)
# display(selected)
